<a href="https://colab.research.google.com/github/deborahpongeluppe/Deborahpongeluppe/blob/main/StartupsSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Startups – apps for reading enthusiasts

This SQL project addresses and analyzes questions posed by a startup looking to invest in apps for avid readers—a market that surged during the coronavirus pandemic as people spent more time at home reading. An initial data review was conducted, followed by the execution of five key tasks: determining the number of books published after January 1, 2000; calculating the number of reviews and the average rating for each book; identifying the publisher that released the highest number of books exceeding 50 pages; finding the author with the highest average rating for books with at least 50 ratings (and separately for those with over 100 ratings); and calculating the average number of reviews among users who have rated more than 50 books.

In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-final-project-db'
}


In [3]:
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)
engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [4]:
query = '''
SELECT * FROM books LIMIT 5;
'''
df = pd.read_sql(query, engine)
print(df)

   book_id  author_id                                              title  \
0        1        546                                       'Salem's Lot   
1        2        465                 1 000 Places to See Before You Die   
2        3        407  13 Little Blue Envelopes (Little Blue Envelope...   
3        4         82  1491: New Revelations of the Americas Before C...   
4        5        125                                               1776   

   num_pages publication_date  publisher_id  
0        594       2005-11-01            93  
1        992       2003-05-22           336  
2        322       2010-12-21           135  
3        541       2006-10-10           309  
4        386       2006-07-04           268  


In [5]:
query = '''
SELECT * FROM authors LIMIT 5;
'''
df = pd.read_sql(query, engine)
print(df)

   author_id                          author
0          1                      A.S. Byatt
1          2  Aesop/Laura Harris/Laura Gibbs
2          3                 Agatha Christie
3          4                   Alan Brennert
4          5        Alan Moore/David   Lloyd


In [6]:
query = '''
SELECT * FROM ratings LIMIT 5;
'''
df = pd.read_sql(query, engine)
print(df)

   rating_id  book_id       username  rating
0          1        1     ryanfranco       4
1          2        1  grantpatricia       2
2          3        1   brandtandrea       5
3          4        2       lorichen       3
4          5        2    mariokeller       2


In [7]:
query = '''
SELECT * FROM reviews LIMIT 5;
'''
df = pd.read_sql(query, engine)
print(df)

   review_id  book_id       username  \
0          1        1   brandtandrea   
1          2        1     ryanfranco   
2          3        2       lorichen   
3          4        3  johnsonamanda   
4          5        3    scotttamara   

                                                text  
0  Mention society tell send professor analysis. ...  
1  Foot glass pretty audience hit themselves. Amo...  
2  Listen treat keep worry. Miss husband tax but ...  
3  Finally month interesting blue could nature cu...  
4  Nation purpose heavy give wait song will. List...  


In [8]:
query = '''
SELECT * FROM publishers LIMIT 5;
'''
df = pd.read_sql(query, engine)
print(df)

   publisher_id                          publisher
0             1                                Ace
1             2                           Ace Book
2             3                          Ace Books
3             4                      Ace Hardcover
4             5  Addison Wesley Publishing Company


In [9]:
query = '''
SELECT COUNT(*)
FROM books
WHERE publication_date > '2000-01-01';
'''
df = pd.read_sql(query, engine)
print(df)

   count
0    819


Based on the analysis, the number of books released after January 1, 2000, is 819.

In [10]:
query = '''
SELECT
    b.title,
    COUNT(r.rating) AS num_avaliacoes,
    AVG(r.rating) AS classificacao_media
FROM books b
JOIN ratings r ON b.book_id = r.book_id
GROUP BY b.book_id, b.title
ORDER BY num_avaliacoes DESC;
'''
df = pd.read_sql(query, engine)
print(df)

                                                 title  num_avaliacoes  \
0                              Twilight (Twilight  #1)             160   
1                  The Hobbit  or There and Back Again              88   
2                               The Catcher in the Rye              86   
3                 Angels & Demons (Robert Langdon  #1)              84   
4    Harry Potter and the Prisoner of Azkaban (Harr...              82   
..                                                 ...             ...   
995                  Naked Empire (Sword of Truth  #8)               2   
996          A Woman of Substance (Emma Harte Saga #1)               2   
997          The Body in the Library (Miss Marple  #3)               2   
998  The Magicians' Guild (Black Magician Trilogy  #1)               2   
999  Disney's Beauty and the Beast (A Little Golden...               1   

     classificacao_media  
0               3.662500  
1               4.125000  
2               3.825581  
3  

Here we can see that the book with the most reviews is *Twilight*, with 160 reviews, followed by *The Hobbit*. When analyzing the average ratings, we see that the books with the most reviews are not the ones with the highest average ratings; we can also observe that the book with the highest average rating—*A Woman of Substance*—has only two reviews.

In [11]:
query = '''
SELECT
    p.publisher,
    COUNT(b.book_id) AS total_livros
FROM books b
INNER JOIN publishers p
    ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY total_livros DESC
LIMIT 1;
'''
df = pd.read_sql(query, engine)
print(df)

       publisher  total_livros
0  Penguin Books            42


In this analysis, we can see that the publisher with the highest number of books exceeding 50 pages is Penguin Books, with approximately 42 books.

In [12]:
query = '''
SELECT
    a.author,
    COUNT(r.rating) AS total_avaliacoes,
    AVG(r.rating) AS classificacao_media
FROM books b
JOIN authors a ON b.author_id = a.author_id
JOIN ratings r ON b.book_id = r.book_id
GROUP BY a.author_id, a.author
HAVING COUNT(r.rating) >= 50
ORDER BY classificacao_media DESC
LIMIT 10;
'''
df = pd.read_sql(query, engine)
print(df)

                              author  total_avaliacoes  classificacao_media
0                     Diana Gabaldon                50             4.300000
1         J.K. Rowling/Mary GrandPré               312             4.288462
2                    Agatha Christie                53             4.283019
3  Markus Zusak/Cao Xuân Việt Khương                53             4.264151
4                     J.R.R. Tolkien               166             4.240964
5           Roald Dahl/Quentin Blake                62             4.209677
6                  Louisa May Alcott                54             4.203704
7                       Rick Riordan                84             4.130952
8                      Arthur Golden                56             4.107143
9                       Stephen King               106             4.009434


This analysis selected the 10 authors with the highest average ratings who have at least 50 reviews. In this context, we can see that Diana Gabaldon has an average rating of 4.30 based on 50 reviews, while J.K. Rowling/Mary GrandPré rank second with an average rating of 4.28 but a total of 312 reviews. Consequently, it is necessary to consider factors beyond just these parameters; a rating based on 312 reviews seems more reliable than one based on 50, and Diana Gabaldon’s rating could potentially drop as she accumulates more reviews. Therefore, the most meaningful approach is to determine the best author based on a combination of high ratings and review volume, rather than relying solely on the average rating.

In [13]:
query = '''
SELECT
    a.author,
    COUNT(r.rating) AS total_avaliacoes,
    AVG(r.rating) AS classificacao_media
FROM books b
JOIN authors a ON b.author_id = a.author_id
JOIN ratings r ON b.book_id = r.book_id
GROUP BY a.author_id, a.author
HAVING COUNT(r.rating) >= 100
ORDER BY classificacao_media DESC
LIMIT 5;
'''

df = pd.read_sql(query, engine)
print(df)

                       author  total_avaliacoes  classificacao_media
0  J.K. Rowling/Mary GrandPré               312             4.288462
1              J.R.R. Tolkien               166             4.240964
2                Stephen King               106             4.009434
3             Nicholas Sparks               111             3.882883
4                   Dan Brown               143             3.741259


Here, the filter was adjusted to a minimum of 100 reviews, yielding more reliable results by raising the total review count threshold; this shifts the top author to J.K. Rowling/Mary GrandPré and places J.R.R. Tolkien second, while the top author from the previous analysis drops out of contention entirely because they had only 50 reviews in total.

In [14]:
query = '''
SELECT
    username,
    COUNT(rating) AS total_avaliacoes
FROM ratings
GROUP BY username
ORDER BY total_avaliacoes DESC
LIMIT 10;
'''
df = pd.read_sql(query, engine)
print(df)

         username  total_avaliacoes
0          paul88                56
1      martinadam                56
2     sfitzgerald                55
3       richard89                55
4  jennifermiller                53
5          xdavis                51
6     lesliegibbs                50
7  vanessagardner                50
8  shermannatalie                50
9     andreaeaton                49


In [15]:
query = '''
SELECT AVG(total_avaliacoes) AS media_avaliacoes_usuarios_ativos
FROM (
    SELECT
        username,
        COUNT(rating) AS total_avaliacoes
    FROM ratings
    GROUP BY username
    HAVING COUNT(rating) > 50
) AS usuarios_ativos;
'''

df = pd.read_sql(query, engine)
print(df)

   media_avaliacoes_usuarios_ativos
0                         54.333333


In [16]:
query = '''
SELECT
    AVG(review_count) AS average_reviews
FROM (
    SELECT
        COUNT(*) AS review_count
    FROM
        reviews
    WHERE
        username IN (
            SELECT
                username
            FROM
                ratings
            GROUP BY
                username
            HAVING
                COUNT(*) > 50  -- Usuários com mais de 50 ratings
        )
    GROUP BY
        username
) AS review_counts;'''
df = pd.read_sql(query, engine)
print(df)

   average_reviews
0        24.333333


The results of this analysis show that the average number of ratings among active users is 54.33; six users rated more than 50 books—with paul88 and martinadam submitting 56 ratings each, sfitzgerald and richard89 submitting 55, jennifermiller submitting 53, and xdavis submitting 51—making them the most active users. As shown above, these "super-active" users (who provided over 50 ratings) wrote an average of 24.3 reviews each; this reveals that the most engaged users tend to write a significant number of reviews, highlighting their importance to the startup.